# 02 · Silver: clean, type and validate

**Pipeline:** bronze → **silver** → gold features → churn model → ARR forecast → retention playbook

* Cast string dates and amounts to proper types
* Drop exact duplicate rows
* Quarantine MRR rows with missing or non-positive amounts (revenue can't be imputed safely)
* Treat negative login counts as a *missing-value sentinel* → `NULL`, keeping the customer-month
  (dropping it would silently remove rows - including churners' final months - from training)
* Enforce invariants with **Delta `CHECK` constraints** and a data-quality suite that fails the job
  on any violation; results are logged to `silver_dq_results`

In [1]:
%pip install -q -r ../requirements.txt

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from revenue_pipeline import config, spark_utils

spark = spark_utils.get_spark()
print(f"Spark {spark.version} | schema: {config.SCHEMA} | Databricks: {spark_utils.on_databricks()}")

Spark 3.5.3 | schema: revenue_forecasting | Databricks: False


In [3]:
silver_customers = spark.sql('''
    SELECT CAST(customer_id AS INT) AS customer_id,
           CAST(signup_date AS DATE) AS signup_date,
           CAST(churn_date AS DATE)  AS churn_date,
           segment, initial_plan, acquisition_channel, contract_type
    FROM bronze_customers
    WHERE customer_id IS NOT NULL''')

silver_mrr = spark.sql('''
    SELECT DISTINCT CAST(customer_id AS INT) AS customer_id,
           CAST(month AS DATE) AS month, plan_type, CAST(mrr AS DOUBLE) AS mrr
    FROM bronze_mrr
    WHERE mrr IS NOT NULL AND NOT isnan(mrr) AND mrr > 0''')

silver_activity = spark.sql('''
    SELECT DISTINCT CAST(customer_id AS INT) AS customer_id,
           CAST(month AS DATE) AS month,
           CASE WHEN logins < 0 THEN NULL ELSE CAST(logins AS INT) END AS logins,
           CAST(support_tickets AS INT) AS support_tickets
    FROM bronze_activity''')

cleaning = pd.DataFrame([
    ("mrr", spark.table("bronze_mrr").count(), spark.table("bronze_mrr").distinct().count(), silver_mrr.count()),
    ("activity", spark.table("bronze_activity").count(), spark.table("bronze_activity").distinct().count(),
     silver_activity.count()),
], columns=["feed", "bronze_rows", "after_dedup", "silver_rows"])
cleaning["duplicates_removed"] = cleaning["bronze_rows"] - cleaning["after_dedup"]
cleaning["quarantined"] = cleaning["after_dedup"] - cleaning["silver_rows"]
print(cleaning.to_string(index=False))
print(f"Sentinel logins set to NULL: {silver_activity.filter('logins IS NULL').count():,}")

    feed  bronze_rows  after_dedup  silver_rows  duplicates_removed  quarantined
     mrr       135280       134875       134732                 405          143
activity       135280       134875       134875                 405            0


Sentinel logins set to NULL: 140


In [4]:
spark_utils.write_table(spark, silver_customers, "silver_customers")
spark_utils.write_table(spark, silver_mrr, "silver_mrr")
spark_utils.write_table(spark, silver_activity, "silver_activity")

CONSTRAINTS = {
    "silver_mrr":       {"mrr_positive": "mrr > 0", "mrr_keys": "customer_id IS NOT NULL AND month IS NOT NULL"},
    "silver_activity":  {"counts_non_negative": "logins IS NULL OR logins >= 0", "tickets_non_negative": "support_tickets >= 0"},
    "silver_customers": {"churn_after_signup": "churn_date IS NULL OR churn_date > signup_date"},
}
for table, rules in CONSTRAINTS.items():
    for name, expr in rules.items():
        spark.sql(f"ALTER TABLE {table} DROP CONSTRAINT IF EXISTS {name}")
        spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} CHECK ({expr})")
print("Silver tables written with Delta CHECK constraints")

Silver tables written with Delta CHECK constraints


In [5]:
from revenue_pipeline import quality

dq = quality.run_checks(spark, raise_on_failure=False)
spark_utils.write_table(spark, dq.assign(run_at=pd.Timestamp.now(tz="UTC")), "silver_dq_results")
print(dq.to_string(index=False))
assert dq["passed"].all(), "Data-quality checks failed - stopping the pipeline"

                                       check  violations  passed
               customers: unique customer_id           0    True
               customers: churn after signup           0    True
            mrr: unique (customer_id, month)           0    True
                       mrr: positive amounts           0    True
                         mrr: known customer           0    True
mrr: no revenue before signup or after churn           0    True
       activity: unique (customer_id, month)           0    True
               activity: non-negative counts           0    True
